In [22]:
import torch
from transformers import NllbTokenizer
from wtpsplit import SaT

from lcm_explo.m2m_100 import M2M100EncoderModel

encoder = M2M100EncoderModel.from_pretrained("cointegrated/SONAR_200_text_encoder_hf")
tokenizer = NllbTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="eng_Latn", tgt_lang="eng_Latn")

sat = SaT("sat-6l")
sat.half().to("mps")

text = """
BioShock est un jeu vidéo de tir en vue à la première personne développé par 2K Boston/2K Australia et conçu par Ken Levine pour l'éditeur américain 2K Games sur le moteur Unreal Engine 2. Il sort sur Xbox 360 et Windows en août 2007. Le titre est porté par 2K Marin et Digital Extremes sur PlayStation 3 en octobre 2008, puis par Feral Interactive sur OS X en février 2009. Il est ensuite porté sur Nintendo Switch en septembre 2016 dans la collection BioShock: The Collection.

Le jeu prend place en 1960. Le personnage incarné par le joueur se nomme Jack. Seul survivant d'un accident aérien en pleine mer, il découvre la ville sous-marine de Rapture construite par le mégalomane milliardaire Andrew Ryan au lendemain de la Seconde Guerre mondiale afin d'y réaliser ses rêves les plus fous d'une société utopique, loin de toute morale extérieure.

Le jeu est présenté par ses développeurs comme le « descendant spirituel » d'un de leurs précédents titres, System Shock 2[8]. L'ambiance est principalement art déco avec une forte inspiration steampunk et dieselpunk.

L'accueil critique du titre a été particulièrement enthousiaste : les journalistes louant particulièrement un univers original et immersif, une narration bien intégrée dans le gameplay, l'intégration de choix moraux en cours de partie, ainsi que la réflexion autour de la pensée objectiviste. Les ventes du jeu atteignent environ 3 millions d'unités écoulées en juin 2009. Ce succès pousse l'éditeur à développer la licence : en février 2010 sort une suite, BioShock 2, dès lors Take-Two Interactive envisage la possibilité d'un film[9],[10]. Le troisième opus, BioShock Infinite sorti en 2013, ne se déroule plus dans la cité sous-marine de Rapture mais dans la ville céleste nommée Columbia. Une compilation des trois opus de la série en version remastérisée, intitulée Bioshock: The Collection, est sorti en 2016 sur Microsoft Windows, PlayStation 4 et Xbox One, puis en 2020 sur Nintendo Switch.
"""

splitted_sample = list(sat.split(text))

sentences = [split for split in splitted_sample if split != ""]

inputs = tokenizer(sentences, padding=True, return_tensors="pt")
with torch.inference_mode():
    encoder_out = encoder(**inputs, pool_last_hidden_state=True)
    embeddings = encoder_out.last_hidden_state.squeeze(1)

print(embeddings.shape)

Python(47355) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


torch.Size([14, 1024])


In [23]:
from transformers.modeling_outputs import BaseModelOutput

from lcm_explo.m2m_100 import M2M100DecoderModel

decoder = M2M100DecoderModel.from_pretrained("cointegrated/SONAR_200_text_decoder_hf")

# Decoding into the original (English) language
generator_out = decoder.generate(
    # passing encoder_outputs is not recommended, because beam search decoding modifies them in place, which is ugly
    # encoder_outputs=enc_out,
    encoder_outputs=BaseModelOutput(last_hidden_state=embeddings.unsqueeze(1)),
    num_beams=5,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
)
text_out = tokenizer.batch_decode(generator_out, skip_special_tokens=True)
print(text_out)  # ["My name is SONAR.", "I can embed the sentences into vector space."]

# Decoding into some other (French) language
generator_out = decoder.generate(
    # passing encoder_outputs is not recommended, because beam search decoding modifies them in place, which is ugly
    # encoder_outputs=enc_out,
    encoder_outputs=BaseModelOutput(last_hidden_state=embeddings.unsqueeze(1)),
    num_beams=5,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("fra_Latn"),
)
text_out = tokenizer.batch_decode(generator_out, skip_special_tokens=True)
print(text_out)  # ['Mon nom est SONAR.', "Je peux intégrer les phrases dans l'espace vectoriel."]

['BioShock is a first-person shooter video game developed by 2K Boston/', 'It released on Xbox 360 and Windows in august 2007.', 'The title was ported by 2K Marin and Digital Extremes on PlayStation 3 in October', 'It is then ported on Nintendo Switch in September 2016 in the BioShock: The', 'Le jeu takes place in 1960.', 'The character embodied by the player calls himself Jack.', 'The only survivor of a plane crash in the sea, he discovers the underwater', 'The game is presented by its developers as the spiritual descendant of one', 'The atmosphere is mainly art deco with a strong inspiration steampunk and diesel', 'The critical reception of the title was particularly enthusiastic: journalists particularly praised', 'The sales of the game reached approximately 3 million units audiences in June 2009.', 'This success prompted the publisher to develop the license: in February 2010 it released a se', 'The third opus, BioShock Infinite released in 2013, takes place no longer in', 'A compil

In [24]:
print(text)


BioShock est un jeu vidéo de tir en vue à la première personne développé par 2K Boston/2K Australia et conçu par Ken Levine pour l'éditeur américain 2K Games sur le moteur Unreal Engine 2. Il sort sur Xbox 360 et Windows en août 2007. Le titre est porté par 2K Marin et Digital Extremes sur PlayStation 3 en octobre 2008, puis par Feral Interactive sur OS X en février 2009. Il est ensuite porté sur Nintendo Switch en septembre 2016 dans la collection BioShock: The Collection.

Le jeu prend place en 1960. Le personnage incarné par le joueur se nomme Jack. Seul survivant d'un accident aérien en pleine mer, il découvre la ville sous-marine de Rapture construite par le mégalomane milliardaire Andrew Ryan au lendemain de la Seconde Guerre mondiale afin d'y réaliser ses rêves les plus fous d'une société utopique, loin de toute morale extérieure.

Le jeu est présenté par ses développeurs comme le « descendant spirituel » d'un de leurs précédents titres, System Shock 2[8]. L'ambiance est princi

In [28]:
print(". ".join(text_out))

BioShock est un jeu de tir à la première personne développé par 2K Boston. Il sort sur Xbox 360 et Windows en août 2007.. Le titre a été porté par 2K Marin et Digital Extremes sur PlayStation 3 en. Il est ensuite porté sur Nintendo Switch en septembre 2016 dans la collection BioShock:. Le jeu a lieu en 1960.. Le personnage incarné par le joueur s'appelle Jack.. Seul survivant d'un accident d'avion en mer, il découvre la. Le jeu est présenté par ses développeurs comme le descendant spirituel de l'. L'ambiance est principalement art déco avec une forte inspiration de steampunk. L'accueil critique du titre a été particulièrement enthousiaste: les journal. Les ventes du jeu atteignent environ 3 millions d'unités écoulées en. Ce succès a poussé l'éditeur à développer la licence: en fé. Le troisième opus, BioShock Infinite sorti en 2013, ne se déroule. Une compilation des trois opus de la série en version remasterisée, int
